# AI Financial Risk Copilot: Multi-Dimensional ISS Scoring Mathematics
### Computational Finance Sandbox for Investor Safety & Cognition Modeling

This Jupyter Notebook demonstrates the advanced mathematical and statistical foundations of the **AI Financial Risk Copilot & Cognition Framework**.

We implement, standardize, and visualize the proprietary **six-category Investor Safety Score (ISS)** equations:
1. **Concentration Risk (CR)** using the Herfindahl-Hirschman Index ($HHI$)
2. **Volatility Risk (VR)** based on asset covariances and active margin leverage multipliers
3. **Liquidity Risk (LR)** evaluating cash and index-anchor margins
4. **Leverage Risk (LEV)** quantifying borrow exposure parameters
5. **Emotional Risk (B)** derived from NLP sentiment active bias coefficients
6. **Diversification Score (DR)** calculating average asset correlation ratios

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set sleek aesthetic styling for matplotlib
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'ggplot')
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.family"] = "sans-serif"

## 1. Quantitative Portfolio Analytics (HHI Concentration & Covariance Volatility)

We define the standard deviations for our four core sandbox assets:
* Tech Giants ($TSLA$, AAPL): $\sigma_{tech} = 18\%$
* Speculative Equities ($GME$, AMC): $\sigma_{meme} = 75\%$
* Meme Cryptocurrencies ($DOGE$, SHIB): $\sigma_{crypto} = 95\%$
* Broad Index Funds (SPY, VOO): $\sigma_{index} = 15\%$

### Volatility Math with Covariance and Leverage
$$\sigma_p = \sqrt{\mathbf{w}^T \mathbf{\Sigma} \mathbf{w}} \times \text{Leverage}$$
Assuming an asset correlation of $\rho = 0.22$.

In [ ]:
ASSET_VOLS = np.array([0.18, 0.75, 0.95, 0.15]) # Tech, Meme, Crypto, Index
CORRELATION = 0.22

def calculate_covariance_matrix(vols, correlation):
    n = len(vols)
    cov_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i == j:
                cov_matrix[i, j] = vols[i] ** 2
            else:
                cov_matrix[i, j] = correlation * vols[i] * vols[j]
    return cov_matrix

COV_MATRIX = calculate_covariance_matrix(ASSET_VOLS, CORRELATION)
print("Asset Covariance Matrix:")
print(pd.DataFrame(COV_MATRIX, columns=['Tech', 'Meme', 'Crypto', 'Index'], index=['Tech', 'Meme', 'Crypto', 'Index']))

## 2. Six-Category Standardization & ISS Composite Equations

We model the core standardizing functions:
* **Concentration Risk**: $CR = HHI \times 100$
* **Volatility Risk**: $VR = \min(100, (\sigma_p / 15\%) \times 50)$
* **Liquidity Risk**: $LR = 100 - (\text{Cash} + \text{Index Weight})$
* **Leverage Risk**: $LEV = (\text{Leverage} - 1.0) \times 50$, capped at 100
* **Emotional Risk**: $B = \text{NLP Score}$ (25 to 100)
* **Diversification Risk**: $DR = 100 - (1 - HHI) \times 100$

$$ISS = 100 - \left( 0.25 \cdot CR + 0.20 \cdot VR + 0.10 \cdot LR + 0.15 \cdot LEV + 0.20 \cdot \mathcal{B} + 0.10 \cdot DR \right)$$

In [ ]:
def calculate_six_category_iss(weights, leverage, cash, b_score):
    # Normalize portfolio weights
    w = np.array(weights) / 100.0
    w = w / np.sum(w)
    
    # 1. Concentration Risk (CR)
    hhi = np.sum(w ** 2)
    cr = hhi * 100.0
    
    # 2. Volatility Risk (VR)
    p_var = np.dot(w.T, np.dot(COV_MATRIX, w))
    p_vol = np.sqrt(p_var) * leverage
    vr = min(100.0, (p_vol / 0.15) * 50.0)
    
    # 3. Liquidity Risk (LR) - VOO index + liquid cash balance inverse
    lr = max(0.0, 100.0 - (cash + weights[3]))
    
    # 4. Leverage Risk (LEV)
    lev = min(100.0, (leverage - 1.0) * 50.0)
    
    # 5. Emotional Risk (B)
    b = b_score
    
    # 6. Diversification Risk (DR)
    dr = hhi * 100.0 # inverse scaling
    
    # Composite Safety Score
    weighted_sum = (0.25 * cr) + (0.20 * vr) + (0.10 * lr) + (0.15 * lev) + (0.20 * b) + (0.10 * dr)
    iss = round(100 - weighted_sum)
    iss = max(0, min(100, iss))
    
    return iss, [cr, vr, lr, lev, b, dr]

# Evaluate specific simulation states (Case Studies)
case_studies = {
    "Case Study 1: Asymmetric Momentum (80% TSLA, 20% BTC)": {
        "weights": [80, 0, 20, 0], "leverage": 1.0, "cash": 5, "b_score": 25
    },
    "Case Study 2: Revenge Margin Trading (40% Tech, 35% GME, 25% Crypto)": {
        "weights": [40, 35, 25, 0], "leverage": 2.0, "cash": 0, "b_score": 95
    },
    "Control Baseline: Anchor Cushion (40% VOO, 40% Tech, 20% Cash)": {
        "weights": [40, 0, 0, 40], "leverage": 1.0, "cash": 20, "b_score": 25
    }
}

print(f"{'Case Study Profile':<70} | {'ISS':<5} | {'CR':<4} | {'VR':<4} | {'LR':<4} | {'LEV':<4} | {'B':<4}")
print("-" * 110)
for name, p in case_studies.items():
    score, cats = calculate_six_category_iss(p["weights"], p["leverage"], p["cash"], p["b_score"])
    print(f"{name:<70} | {score:<5} | {round(cats[0]):<4} | {round(cats[1]):<4} | {round(cats[2]):<4} | {round(cats[3]):<4} | {round(cats[4]):<4}")

## 3. Visualizations: Cognitive Scoring Breakdown

We plot a multi-bar chart comparing the six risk vectors between a speculative revenge trader (Case Study 2) and a rational anchor investor (Control Baseline), illustrating the exact dimensional shifts.

In [ ]:
# Compute vectors
_, cs2_cats = calculate_six_category_iss([40, 35, 25, 0], leverage=2.0, cash=0, b_score=95)
_, ctrl_cats = calculate_six_category_iss([40, 0, 0, 40], leverage=1.0, cash=20, b_score=25)

categories = ['Concentration', 'Volatility', 'Liquidity', 'Leverage', 'Behavioral Risk', 'Diversification']
x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots()
rects1 = ax.bar(x - width/2, cs2_cats, width, label='Revenge Speculator (ISS: 23)', color='crimson')
rects2 = ax.bar(x + width/2, ctrl_cats, width, label='Anchor Investor (ISS: 83)', color='emerald' if 'emerald' in plt.rcParams['axes.prop_cycle'].by_key()['color'] else 'teal')

ax.set_ylabel('Standardized Risk Score (0-100)')
ax.set_title('Risk Profile Comparison: Speculative Revenge vs Rational Base', fontsize=13, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylim(0, 110)
ax.legend(frameon=True, facecolor='#ffffff')

plt.tight_layout()
plt.show()